In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML

# ==========================================
# 1. CG STABILITY MODEL & SAFE GLIDER VALIDATION
# ==========================================
NEUTRAL_POINT_MAC = 0.35  # 35% MAC reference neutral point

def evaluate_cg_stability(cg_pct, val_mode, manual_peak_deg, duration_s=8.0, dt=0.02):
    cg_mac = cg_pct / 100.0
    # Static margin = (X_np - X_cg) / MAC  (positive => statically stable)
    static_margin = NEUTRAL_POINT_MAC - cg_mac

    n = int(duration_s / dt) + 1
    t = np.linspace(0.0, duration_s, n)

    # Initial pitch disturbance after safe glider hand-launch [deg]
    init_pitch = 8.0

    # Effective restoring stiffness & damping from static margin
    # If static_margin <= 0, restoring stiffness drops/turns negative -> instability/oscillation
    k_rest = max(static_margin * 14.0, -2.5)
    if static_margin > 0.08:
        damping = 1.65 + 4.0 * static_margin
        omega = np.sqrt(max(k_rest, 0.2))
        zeta = np.clip(damping / (2.0 * omega), 0.15, 1.35)
        if zeta < 1.0:
            wd = omega * np.sqrt(max(1.0 - zeta**2, 0.02))
            model_pitch = init_pitch * np.exp(-zeta * omega * t) * np.cos(wd * t)
        else:
            model_pitch = init_pitch * np.exp(-1.8 * t)
        stability_label = "STABLE (Positive Static Margin)"
        badge_color = "#16a34a"
    elif static_margin > -0.02:
        # Marginal / lightly damped oscillation
        model_pitch = init_pitch * np.exp(-0.18 * t) * np.cos(2.4 * t)
        stability_label = "MARGINALLY STABLE (Low Margin)"
        badge_color = "#eab308"
    else:
        # Aft CG past neutral point -> divergent / heavy pitch oscillation
        growth_or_osc = np.clip(np.exp(0.28 * t), 1.0, 3.2) * np.cos(3.2 * t)
        model_pitch = np.clip(init_pitch * growth_or_osc, -35.0, 35.0)
        stability_label = "UNSTABLE (Aft CG / Negative Margin)"
        badge_color = "#dc2626"

    # Safe glider test comparison series
    if val_mode == "Manual Glider Test Peak [deg]":
        decay_env = np.exp(-0.35 * t) * np.cos(2.5 * t)
        max_abs = max(np.max(np.abs(decay_env)), 1e-6)
        test_pitch = manual_peak_deg * (decay_env / max_abs)
        test_source = f"Manual safe glider test peak ({manual_peak_deg:+.1f}°)"
    else:
        # Simulated safe glider test closely follows model with minor real-world variation
        test_pitch = model_pitch * 0.94 + 0.35 * np.sin(3.5 * t) * np.exp(-0.25 * t)
        test_source = "Simulated safe glider hand-launch test"

    rmse = float(np.sqrt(np.mean((test_pitch - model_pitch) ** 2)))
    max_model_amp = float(np.max(np.abs(model_pitch)))
    max_test_amp = float(np.max(np.abs(test_pitch)))

    return pd.DataFrame({
        "time_s": t,
        "model_pitch_deg": model_pitch,
        "test_pitch_deg": test_pitch,
    }), {
        "cg_pct": cg_pct,
        "static_margin": static_margin,
        "stability_label": stability_label,
        "badge_color": badge_color,
        "rmse": rmse,
        "max_model_amp": max_model_amp,
        "max_test_amp": max_test_amp,
        "test_source": test_source,
    }

def build_cg_why_explanation(stats):
    sm = stats["static_margin"]
    if sm > 0.08:
        cg_reason = f"CG at {stats['cg_pct']:.0f}% MAC sits ahead of the 35% MAC neutral point (Static Margin = {sm:+.2f}), creating a restoring nose-down tendency after disturbance"
    elif sm > -0.02:
        cg_reason = f"CG at {stats['cg_pct']:.0f}% MAC is near the 35% MAC neutral point (Static Margin = {sm:+.2f}), leaving weak restoring margin and lingering oscillations"
    else:
        cg_reason = f"CG at {stats['cg_pct']:.0f}% MAC moves aft of the 35% MAC neutral point (Static Margin = {sm:+.2f}), reducing pitch stiffness and causing unstable/oscillatory behavior"

    val_reason = f"the {stats['test_source'].lower()} shows a peak amplitude of {stats['max_test_amp']:.1f}° vs. model {stats['max_model_amp']:.1f}° (RMSE = {stats['rmse']:.2f}°)"
    return f"<b>Why This Result Happened:</b> {cg_reason}; and {val_reason}."

# ==========================================
# 2. 2D GLIDER PROFILE + VALIDATION PLOT
# ==========================================
def draw_cg_validation(cg_pct, val_mode, manual_peak_deg):
    df, stats = evaluate_cg_stability(cg_pct, val_mode, manual_peak_deg)
    why_html = build_cg_why_explanation(stats)

    plt.style.use("seaborn-v0_8-whitegrid")
    fig, (ax_glider, ax_val) = plt.subplots(1, 2, figsize=(13.8, 4.7), gridspec_kw={"width_ratios": [1.0, 1.45]})

    # --- LEFT: 2D Glider Side View with CG & Neutral Point ---
    # Fuselage & tail boom
    ax_glider.plot([-1.1, 1.0], [0.0, 0.0], color="#334155", lw=3.5)
    # Main wing chord representation around x = 0.0
    ax_glider.plot([-0.22, 0.22], [0.06, 0.06], color="#2563eb", lw=4.5, label="Main Wing")
    # Horizontal tail at x = -0.95
    ax_glider.plot([-1.05, -0.85], [0.16, 0.04], color="#64748b", lw=3.0, label="Tailplane")

    # Map % MAC [15%..55%] to x [-0.25..+0.25]
    def mac_to_x(pct):
        return ((pct - 35.0) / 20.0) * 0.25

    x_np = mac_to_x(NEUTRAL_POINT_MAC * 100.0)
    x_cg = mac_to_x(cg_pct)

    ax_glider.scatter([x_np], [0.04], color="#0284c7", s=80, zorder=6, label="Neutral Point (35% MAC)")
    ax_glider.scatter([x_cg], [0.0], color=stats["badge_color"], s=95, marker="D", zorder=7, label=f"CG ({cg_pct:.0f}% MAC)")

    ax_glider.set_title(f"2D Glider Layout (Static Margin: {stats['static_margin']:+.2f})", fontsize=10.8, fontweight="bold", pad=10)
    ax_glider.set_xlim(-1.2, 1.15)
    ax_glider.set_ylim(-0.55, 0.55)
    ax_glider.set_aspect("equal", adjustable="box")
    ax_glider.axis("off")
    ax_glider.legend(loc="lower center", ncol=2, fontsize=7.8, frameon=True, facecolor="white")

    # --- RIGHT: Model vs. Safe Glider Test Pitch Response ---
    ax_val.plot(df["time_s"], df["model_pitch_deg"], color="#2563eb", lw=2.2, label="Stability Model Pitch [deg]")
    ax_val.plot(df["time_s"], df["test_pitch_deg"], color="#ea580c", lw=1.9, ls="--", label="Safe Glider Test Pitch [deg]")
    ax_val.axhline(0.0, color="#64748b", lw=1.0)

    ax_val.set_title("Stability Model vs. Safe Glider Test Validation", fontsize=11, fontweight="bold", pad=10)
    ax_val.set_xlabel("Time After Launch [s]", fontsize=9.5)
    ax_val.set_ylabel("Pitch Angle [deg]", fontsize=9.5)
    ax_val.legend(loc="upper right", fontsize=8.3, frameon=True, facecolor="white")

    plt.tight_layout()
    plt.show()

    display(HTML(f"""
    <div style="
        width:100%;box-sizing:border-box;
        background:linear-gradient(135deg,#0f172a 0%,#1e293b 100%);
        border:1px solid #334155;border-left:5px solid {stats['badge_color']};
        border-radius:8px;padding:14px 18px;color:#f8fafc;
        font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',Roboto,sans-serif;margin-top:8px;
    ">
        <div style="font-size:13px;">
            <b>Stability Verdict:</b> <span style="color:#ffffff;font-weight:700;">{stats['stability_label']}</span> &nbsp;|&nbsp;
            <b>CG:</b> {stats['cg_pct']:.0f}% MAC &nbsp;|&nbsp;
            <b>Static Margin:</b> {stats['static_margin']:+.2f} &nbsp;|&nbsp;
            <b>Validation RMSE:</b> {stats['rmse']:.2f}°
        </div>
        <div style="font-size:12.5px;color:#e2e8f0;margin-top:6px;line-height:1.55;">
            {why_html}
        </div>
    </div>
    """))

# ==========================================
# 3. INTERACTIVE ENGLISH UI
# ==========================================
lbl_style = {"description_width": "165px"}
ctrl_layout = widgets.Layout(width="350px")

w_cg    = widgets.FloatSlider(value=25.0, min=15.0, max=50.0, step=1.0, description="CG Position [% MAC]:", readout_format=".0f", style=lbl_style, layout=ctrl_layout)
w_vmode = widgets.Dropdown(options=["Simulated Safe Glider Test", "Manual Glider Test Peak [deg]"], value="Simulated Safe Glider Test", description="Validation Source:", style=lbl_style, layout=ctrl_layout)
w_mpeak = widgets.FloatSlider(value=7.5,  min=-25.0, max=25.0, step=0.5, description="Manual Test Peak [deg]:", readout_format="+.1f", style=lbl_style, layout=ctrl_layout)

btn_cg = widgets.Button(description="Run CG Validation", button_style="primary", icon="check-circle", layout=widgets.Layout(width="215px", height="36px"))
out_cg = widgets.Output(layout=widgets.Layout(width="100%"))

def refresh_cg(*args):
    with out_cg:
        clear_output(wait=True)
        draw_cg_validation(w_cg.value, w_vmode.value, w_mpeak.value)

btn_cg.on_click(refresh_cg)
for c in [w_cg, w_vmode, w_mpeak]:
    c.observe(refresh_cg, names="value")

header_cg = widgets.HTML("""
    <div style="
        width:100%;box-sizing:border-box;
        background:linear-gradient(135deg,#0f172a 0%,#1e293b 100%);
        border:1px solid #334155;border-left:4px solid #38bdf8;
        border-radius:8px;padding:12px 16px;margin-bottom:10px;
        font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',Roboto,sans-serif;
    ">
        <div style="font-size:17px;font-weight:700;color:#f8fafc;">CG Position Stability Model &amp; Safe Glider Validation</div>
        <div style="font-size:12px;color:#94a3b8;margin-top:2px;">
            Model how center-of-gravity location shapes static stability and validate against a safe glider test
        </div>
    </div>
""")

controls_cg = widgets.VBox([
    widgets.HBox([
        widgets.VBox([w_cg, w_vmode], layout=widgets.Layout(margin="0 28px 0 0", gap="8px")),
        widgets.VBox([w_mpeak, btn_cg], layout=widgets.Layout(gap="8px"))
    ], layout=widgets.Layout(align_items="flex-start", flex_flow="row wrap"))
], layout=widgets.Layout(
    width="100%", padding="14px 16px", border="1px solid #334155",
    border_radius="8px", background_color="#1e293b", margin="0 0 12px 0"
))

display(header_cg, controls_cg, out_cg)
refresh_cg()

HTML(value='\n    <div style="\n        width:100%;box-sizing:border-box;\n        background:linear-gradient(…

Output(layout=Layout(width='100%'))